## How `kube-proxy` actually routes traffic

The Service's ClusterIP is a *fiction* — no process listens on `10.96.42.7`. So what makes the packet arrive at a pod? **`kube-proxy`.** It runs as a DaemonSet on every node, watches the API server for Services and Endpoints, and programs the **node's kernel networking** so packets to a Service IP get rewritten to a real pod IP.

Three modes, two you'll meet:

- **`iptables`** *(long-time default)* — writes rules that DNAT packets for the ClusterIP to a random endpoint. Simple, in-kernel, no data-path process — but rule count grows with Services × endpoints, slowing updates in huge clusters.
- **`IPVS`** — programs the kernel's IP Virtual Server: hash-based lookup instead of linear iteration, richer algorithms (round-robin, least-conn, source-hash). Faster at large scale.
- **`nftables`** — the modern packet-filter framework; the direction for new clusters.

What matters day to day:

- **Debugging** — `iptables -t nat -L -n` on a node shows the rules kube-proxy wrote.
- **Service mesh** — Istio/Linkerd sidecars intercept traffic *before* kube-proxy; they sit on top of it, not instead.
- **Scale** — past a few thousand Services, switch to IPVS or nftables.

The crucial misconception to kill: **kube-proxy is *not* in the data path of an established connection.** It only sets up the *rules*; packets then flow through the kernel directly, pod to pod, no userspace hop. That's why Service IPs are essentially free at runtime. On our map this is the **kube-proxy** component on the worker node wiring the **Service** onto the Pods.